# 06 - Preprocessing pipeline smoke test

Validates the PIPELINE (split -> scaling -> fit), not the science - the honest
model evaluation lives in notebook 08.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

from qsv.paths import MAIN_DATASET, MODELS_DIR
from qsv.preprocessing import prepare_features_and_target, split_data, scale_features

df = pd.read_csv(MAIN_DATASET)
random_state = 42

# norm_squared excluded by default - see notebooks 07/08 for the full leakage story
X, y = prepare_features_and_target(df, include_norm_squared=False)
print(f"Dataset: {df.shape} | features: {X.shape}")

Dataset: (10000, 11) | features: (10000, 8)


In [2]:
X_train, X_val, X_test, y_train, y_val, y_test = split_data(X, y, random_state=random_state)
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Valid share per split: {y_train.mean():.1%} / {y_val.mean():.1%} / {y_test.mean():.1%}")

Train: 6000 | Val: 2000 | Test: 2000
Valid share per split: 50.0% / 50.0% / 50.0%


In [3]:
# Baseline logistic regression on standardized features.
X_train_s, X_val_s, X_test_s, scaler = scale_features(
    X_train, X_val, X_test, method="standard", save_scaler=False
)

model = LogisticRegression(max_iter=2000, random_state=random_state)
model.fit(X_train_s, y_train)
print(f"Validation accuracy: {accuracy_score(y_val, model.predict(X_val_s)):.4f}")
print("(low is expected: raw coordinates only, non-linear boundary - see notebook 08)")

Validation accuracy: 0.5670
(low is expected: raw coordinates only, non-linear boundary - see notebook 08)


In [4]:
import joblib

test_accuracy = accuracy_score(y_test, model.predict(X_test_s))
print(f"Test accuracy: {test_accuracy:.4f}")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
model_path = MODELS_DIR / "baseline_model.joblib"
joblib.dump(model, model_path)
print(f"Model saved: {model_path.name} ({model_path.stat().st_size} bytes)")
print("Preprocessing pipeline validated end to end.")

Test accuracy: 0.5435
Model saved: baseline_model.joblib (1263 bytes)
Preprocessing pipeline validated end to end.
